In [1]:
import os, warnings
from getpass import getpass

import requests
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_ollama import ChatOllama

from langchain_core.prompts import PromptTemplate  
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

warnings.filterwarnings("ignore")


/var/folders/t3/97hgmq6x6mg3dybs2fbsfcqr0000gn/T/ipykernel_48758/2728386894.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/Users/guane/Documentos/GlogalLogic/AI-course/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [ ]:
urls = ['https://arxiv.org/pdf/2306.06031v1.pdf','https://arxiv.org/pdf/2306.12156v1.pdf','https://arxiv.org/pdf/2306.14289v1.pdf','https://arxiv.org/pdf/2305.10973v1.pdf','https://arxiv.org/pdf/2306.13643v1.pdf']
ml_papers = []

for i, url in enumerate(urls):
    
    filename = f'/Users/guane/Documentos/GlogalLogic/AI-course/data/Papers/paper{i+1}.pdf'

    if not os.path.exists(filename):
        response = requests.get(url)
        with open(filename, 'wb') as f:
            f.write(response.content)
        print(f'Descargado {filename}')
    else:
        print(f'{filename} ya existe, cargando desde el disco.')

    loader = PyPDFLoader(filename)
    data = loader.load()
    ml_papers.extend(data)

/Users/guane/Documentos/GlogalLogic/AI-course/data/Papers/paper1.pdf ya existe, cargando desde el disco.
/Users/guane/Documentos/GlogalLogic/AI-course/data/Papers/paper2.pdf ya existe, cargando desde el disco.
/Users/guane/Documentos/GlogalLogic/AI-course/data/Papers/paper3.pdf ya existe, cargando desde el disco.
/Users/guane/Documentos/GlogalLogic/AI-course/data/Papers/paper4.pdf ya existe, cargando desde el disco.
/Users/guane/Documentos/GlogalLogic/AI-course/data/Papers/paper5.pdf ya existe, cargando desde el disco.


In [3]:
print(type(ml_papers), len(ml_papers))

<class 'list'> 57


In [4]:
print(ml_papers[3])

page_content='Figure 1: FinGPT Framework.
4.1 Data Sources
The first stage of the FinGPT pipeline involves the collec-
tion of extensive financial data from a wide array of online
sources. These include, but are not limited to:
• Financial news: Websites such as Reuters, CNBC, Yahoo
Finance, among others, are rich sources of financial news
and market updates. These sites provide valuable informa-
tion on market trends, company earnings, macroeconomic
indicators, and other financial events.
• Social media: Platforms such as Twitter, Facebook, Red-
dit, Weibo, and others, offer a wealth of information in
terms of public sentiment, trending topics, and immediate
reactions to financial news and events.
• Filings: Websites of financial regulatory authorities, such
as the SEC in the United States, offer access to company
filings. These filings include annual reports, quarterly earn-
ings, insider trading reports, and other important company-
specific information. Official websites of stock e

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
    length_function=len
)

documents = text_splitter.split_documents(ml_papers)

In [6]:
print(len(documents), documents[10])

211 page_content='highly volatile, changing rapidly in response to news events
or market movements.
Trends, often observable through websites like Seeking
Alpha, Google Trends, and other finance-oriented blogs and
forums, offer critical insights into market movements and in-
vestment strategies. They feature:
• Analyst perspectives: These platforms provide access to
market predictions and investment advice from seasoned
financial analysts and experts.
• Market sentiment: The discourse on these platforms can
reflect the collective sentiment about specific securities,
sectors, or the overall market, providing valuable insights
into the prevailing market mood.
• Broad coverage: Trends data spans diverse securities and
market segments, offering comprehensive market coverage.
Each of these data sources provides unique insights into
the financial world. By integrating these diverse data types,
financial language models like FinGPT can facilitate a com-
prehensive understanding of financial m

In [ ]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = Chroma.from_documents(
    documents=documents, 
    embedding=embeddings
    )

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
    )


In [8]:
llm = ChatOllama(model="llama3.2:1b",
                temperature=0)


In [9]:
prompt = """\
Answer the question using the provided context.

Context:
{context}

Question:
{question}
"""

prompt_template = PromptTemplate.from_template(prompt)



In [10]:
print(type(retriever))

<class 'langchain_core.vectorstores.base.VectorStoreRetriever'>


In [13]:
# 1️⃣ Función que formatea los documentos (ya la tienes)
def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

# 2️⃣ Encadenar todo correctamente
chain = (
    {
        "context": (lambda x: x["question"]) | retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | prompt_template
    | llm
    | StrOutputParser()
)



In [16]:
question = "Que es FinGPT?"
respuesta = chain.invoke({"question": question})
print(respuesta)


FinGPT se refiere a un modelo de inteligencia artificial (LLM) diseñado para aplicaciones financieras, que ofrece tutoriales y aplicaciones prácticas para tareas como asesoramiento robótico financiero, trading numérico y desarrollo de aplicaciones en línea.
